In [1]:
import pandas as pd  # Para manipulación de datos
import matplotlib.pyplot as plt  # Para visualización
import seaborn as sns  # Para gráficos más estilizados
import numpy as np
import os

In [2]:
# Cargar los datos
csv_file_path = r"../../data/limpieza/cleaned_data.csv"
df = pd.read_csv(csv_file_path)

df.head()

,timestamp,open_binance,high_binance,low_binance,close_binance,volumen_binance,trades_binance,var_diaria_binance,rango_dia_binance,volatilidad_50_binance,...,momentum_200_dxy,SMA_50_dxy,EMA_50_dxy,SMA_200_dxy,EMA_200_dxy,RSI_dxy,pct_cambio_7d,tendencia_futura_7d,pct_cambio_30d,tendencia_futura_30d
0,2023-01-31,22827.38,23320.00,22714.77,23125.13,264649.34909,6798411.0,1.309814,605.23,2664.153492,...,-5.515454,103.303599,103.384715,107.431900,105.527955,40.489557,NaN,0,NaN,0
1,2023-02-01,23125.13,23812.66,22760.23,23732.66,310790.42271,7863213.0,2.627142,1052.43,2742.210819,...,-6.130018,103.248399,103.299825,107.398850,105.485090,29.528127,NaN,0,NaN,0
2,2023-02-02,23731.41,24255.00,23363.27,23488.94,364177.20751,9102300.0,-1.026939,891.73,2805.431757,...,-5.436804,103.207999,103.239047,107.369600,105.447924,40.054093,NaN,0,NaN,0
3,2023-02-03,23489.33,23715.70,23204.62,23431.90,332571.02904,8007846.0,-0.242838,511.08,2855.447164,...,-4.144551,103.175199,103.226536,107.347350,105.422771,55.762488,NaN,0,NaN,0
4,2023-02-04,23431.90,23587.78,23253.96,23326.84,166126.47295,4852404.0,-0.448363,333.82,2883.984445,...,-3.305837,103.144266,103.223665,107.329717,105.400190,58.119508,NaN,0,NaN,0


# Agrupar variables

In [3]:
sufijos = [
    "_binance",
    "_fg",
    "_gtrends",
    "_hashrate",
    "_mDbinance",
    "_sp500",
    "_nasdaq",
    "_dxy",
]

# Crear un diccionario para almacenar las columnas separadas
columnas_por_sufijo = {sufijo: [] for sufijo in sufijos}

# Recorrer las columnas y clasificarlas
for col in df.columns:
    for sufijo in sufijos:
        if col.endswith(sufijo):
            columnas_por_sufijo[sufijo].append(col)
            break

# Ahora tienes las columnas organizadas por sufijo
suma = 5 # timestamp, 4 columnas respuesta
for sufijo, columnas in columnas_por_sufijo.items():
    print(f"\nSufijo {sufijo} ({len(columnas)}):")
    print(columnas)
    suma += len(columnas)

assert suma == df.shape[1], f"Suma: {suma}, columnas:{df.shape[1]}"


Sufijo _binance (17):
['open_binance', 'high_binance', 'low_binance', 'close_binance', 'volumen_binance', 'trades_binance', 'var_diaria_binance', 'rango_dia_binance', 'volatilidad_50_binance', 'momentum_50_binance', 'volatilidad_200_binance', 'momentum_200_binance', 'SMA_50_binance', 'EMA_50_binance', 'SMA_200_binance', 'EMA_200_binance', 'RSI_binance']

Sufijo _fg (1):
['fear_greed_index_fg']

Sufijo _gtrends (1):
['bitcoin_gtrends']

Sufijo _hashrate (5):
['hashrate_hashrate', 'SMA_50_hashrate', 'SMA_200_hashrate', 'EMA_50_hashrate', 'EMA_200_hashrate']

Sufijo _mDbinance (17):
['open_mDbinance', 'high_mDbinance', 'low_mDbinance', 'close_mDbinance', 'volumen_mDbinance', 'trades_mDbinance', 'var_diaria_mDbinance', 'rango_dia_mDbinance', 'volatilidad_50_mDbinance', 'momentum_50_mDbinance', 'volatilidad_200_mDbinance', 'momentum_200_mDbinance', 'SMA_50_mDbinance', 'EMA_50_mDbinance', 'SMA_200_mDbinance', 'EMA_200_mDbinance', 'RSI_mDbinance']

Sufijo _sp500 (16):
['close_sp500', 'high_s

# Eliminación de variables redundantes o con información duplicada


## 1. Binance – precios OHLC
**Eliminar:**
- open_binance
- high_binance
- low_binance
- volumen_binance: se calcula close_binance * volume_binance

**Dejar:**
- close_binance
- volume_binance
- trades_binance
- rango_dia_binance: diferencia de high y low
- var_diaria_binance: cambio porcentual entre un valor y el valor inmediatamente anterior (en porcentaje)

In [4]:
df.drop(columns=['open_binance', 'high_binance', 'low_binance', 'volumen_binance'], inplace=True)

## 2. Binance - Medias móviles
Mantener una combinación de corto y largo plazo, evitando duplicar SMA y EMA para ambos.

**Eliminar:**
- SMA_50_binance
- EMA_200_binance

**Dejar:**
- EMA_50_binance
- SMA_200_binance

In [5]:
df.drop(columns=['SMA_50_binance', 'EMA_200_binance'], inplace=True)

## 3. Binance - Otras varibles
Conservaremos las variables de volatilidad, momentum y RSI

## 4. Indicadores
Conservaremos los indicadores: `fear_greed_index_fg` y `bitcoin_gtrends`

## 5. Hashrate
Conservaremos la variable `hashrate_hashrate` y mantendremos una combinación de corto y largo plazo para las medias móviles como en Binance

**Eliminar:**
- SMA_50_hashrate
- EMA_200_hashrate

**Dejar:**
- EMA_50_hashrate
- SMA_200_hashrate

In [6]:
df.drop(columns=['SMA_50_hashrate', 'EMA_200_hashrate'], inplace=True)

## 6. Mercado Derivados Binance, S&P 500, Nasdaq, DXY
Al compartir las mismas variables que Binance, repetiremos el proceso:

**Eliminar:**
- open
- high
- low
- SMA_50
- EMA_200

**Dejar:**
- close
- volume
- trades
- var_diaria
- rango dia
- volatilidad (50/200)
- momentum (50/200)
- SMA_200
- EMA_50

In [7]:
# mDbinance
df.drop(columns=[
    'open_mDbinance',
    'high_mDbinance',
    'low_mDbinance',
    'SMA_50_mDbinance',
    'EMA_200_mDbinance'
        ],
        inplace=True)

# S&P 500, Nasdaq, DXY
sufijos = ['_sp500', '_nasdaq', '_dxy']
columnas = ['open', 'High', 'Low', 'SMA_50', 'EMA_200']
columnas_a_eliminar = [col + suf for col in columnas for suf in sufijos]

df.drop(columns=columnas_a_eliminar, inplace=True)

KeyError: "['Open_sp500', 'Open_nasdaq', 'Open_dxy', 'High_sp500', 'High_nasdaq', 'High_dxy', 'Low_sp500', 'Low_nasdaq', 'Low_dxy'] not found in axis"

## 7. Varibles respuesta categóricas
Para mostrar la creación de la variable respuesta gráficamente se creó una variable categórica auxiliar que indicaba la tendencia alcista o bajista del bitcoin. Como no se va a realizar problemas de clasificación se eliminarán estas variables

In [ ]:
df.drop(columns=['tendencia_futura_7d', 'tendencia_futura_30d'], inplace=True)

## 8. Conjunto de datos para 7 días
No se eliminará ninguna otra variable salvo la variable respuesta para 30 días

In [ ]:
df_7d = df.drop(columns=['pct_cambio_30d'])

output_path = r"../../data/limpieza/dataset_7d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_7d.to_csv(output_path, index=False)

## 9. Conjunto de datos para 30 días
No se eliminará ninguna otra variable salvo la variable respuesta para 7 días

In [ ]:
df_30d = df.drop(columns=['pct_cambio_7d'])

output_path = r"../../data/limpieza/dataset_30d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_30d.to_csv(output_path, index=False)

In [ ]:
df.head()